# 04 — Barchart WTI full-history futures curve

Strategy:
- full history from 2015;
- F1–F6 plus F12;
- F1 probe first, so trial requests are not wasted;
- dailyNearest;
- expiration roll;
- daysToExpiration=0;
- no back adjustment.


In [ ]:
from __future__ import annotations

import os
from datetime import date
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd

from src.data.barchart_client import (
    BarchartClient,
    BarchartHistoryRequest,
    WTI_NEARBYS,
)


def find_project_root(
    start: Path | None = None,
) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate project root."
    )


def load_env_value(
    path: Path,
    key: str,
) -> str:
    if not path.exists():
        return ""

    for raw_line in path.read_text(
        encoding="utf-8"
    ).splitlines():
        line = raw_line.strip()

        if not line or line.startswith("#"):
            continue

        prefix = f"{key}="

        if line.startswith(prefix):
            return (
                line[len(prefix):]
                .strip()
                .strip('"')
                .strip("'")
            )

    return ""


ROOT = find_project_root()
RAW_DIR = ROOT / "data" / "raw" / "market"
PROCESSED_DIR = ROOT / "data" / "processed" / "market"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

api_key = (
    os.getenv("BARCHART_API_KEY", "").strip()
    or load_env_value(
        ROOT / ".env",
        "BARCHART_API_KEY",
    )
)

if not api_key:
    raise RuntimeError(
        "Set BARCHART_API_KEY in project .env."
    )

barchart = BarchartClient(api_key)


In [ ]:
FETCH_START = date(2015, 1, 1)

FETCH_END_EXCLUSIVE = pd.Timestamp.now(
    tz=ZoneInfo("America/New_York")
).date()

print(
    "requested range:",
    FETCH_START,
    "->",
    FETCH_END_EXCLUSIVE,
    "(end exclusive)",
)
print("nearby maturities:", WTI_NEARBYS)


## Probe F1 only

This spends one API request. If the trial does not reach 2015,
stop before requesting the remaining six maturities.


In [ ]:
f1_probe = barchart.fetch_wti_nearby(
    BarchartHistoryRequest(
        nearby=1,
        start_date=FETCH_START,
        end_date=FETCH_END_EXCLUSIVE,
    )
)

probe_first_day = f1_probe["trading_day"].min()
probe_last_day = f1_probe["trading_day"].max()

print("F1 rows:", len(f1_probe))
print(
    "F1 returned range:",
    probe_first_day,
    "->",
    probe_last_day,
)
print(
    "F1 contracts:",
    f1_probe["symbol"].nunique(),
)

display(f1_probe.head())


In [ ]:
FULL_HISTORY_LATEST_ACCEPTABLE_START = (
    pd.Timestamp("2015-01-09")
)

assert (
    probe_first_day
    <= FULL_HISTORY_LATEST_ACCEPTABLE_START
), (
    "Trial did not return full 2015 history. "
    f"First returned date: {probe_first_day.date()}. "
    "Stop here before spending more requests."
)

print(
    "Full-history probe passed. "
    "Safe to request remaining nearbys."
)


## Fetch F2–F6 + F12

F1 is reused from the probe. Total requests: 7.


In [ ]:
barchart_long = barchart.fetch_wti_nearbys(
    nearbys=WTI_NEARBYS,
    start_date=FETCH_START,
    end_date=FETCH_END_EXCLUSIVE,
    preloaded={1: f1_probe},
)

summary = (
    barchart_long.groupby("nearby")
    .agg(
        rows=("trading_day", "size"),
        first_day=("trading_day", "min"),
        last_day=("trading_day", "max"),
        contracts=("symbol", "nunique"),
        missing_close=(
            "close",
            lambda s: s.isna().sum(),
        ),
        missing_volume=(
            "volume",
            lambda s: s.isna().sum(),
        ),
        missing_open_interest=(
            "open_interest",
            lambda s: s.isna().sum(),
        ),
    )
)

display(summary)


In [ ]:
history_start_by_nearby = (
    barchart_long.groupby("nearby")[
        "trading_day"
    ].min()
)

late_starts = history_start_by_nearby[
    history_start_by_nearby
    > FULL_HISTORY_LATEST_ACCEPTABLE_START
]

assert late_starts.empty, (
    "Some nearby series do not reach 2015:\n"
    f"{late_starts}"
)

assert not barchart_long.duplicated(
    ["trading_day", "nearby"]
).any()

assert barchart_long["symbol"].notna().all()

print("All nearby series reach 2015.")


## Inspect actual contract rolls


In [ ]:
contract_rolls = (
    barchart_long[
        ["trading_day", "nearby", "symbol"]
    ]
    .sort_values(["nearby", "trading_day"])
    .assign(
        previous_symbol=lambda frame: (
            frame.groupby("nearby")[
                "symbol"
            ].shift()
        )
    )
)

contract_rolls = contract_rolls[
    contract_rolls["symbol"]
    != contract_rolls["previous_symbol"]
].copy()

display(contract_rolls.head(60))
print("roll observations:", len(contract_rolls))


## Build curve and spreads


In [ ]:
barchart_curve = barchart.build_close_curve(
    barchart_long
)

required_price_columns = [
    "wti_f1",
    "wti_f2",
    "wti_f3",
    "wti_f4",
    "wti_f5",
    "wti_f6",
    "wti_f12",
]

display(
    barchart_curve[
        [
            "trading_day",
            *required_price_columns,
            "spread_f1_f2",
            "spread_f2_f3",
            "spread_f1_f3",
            "spread_f1_f6",
            "spread_f1_f12",
        ]
    ].head()
)


In [ ]:
barchart_symbols = barchart.build_symbol_curve(
    barchart_long
)

display(barchart_symbols.head())


## Curve QA


In [ ]:
complete_share = (
    barchart_curve[
        required_price_columns
    ]
    .notna()
    .all(axis=1)
    .mean()
)

print(
    "complete F1-F6/F12 share:",
    round(complete_share, 4),
)

assert complete_share > 0.98, (
    "Too many incomplete WTI curve rows: "
    f"{complete_share:.4f}"
)

assert not barchart_curve[
    "trading_day"
].duplicated().any()

print("Curve QA passed.")


## Save raw and processed outputs


In [ ]:
raw_path = (
    RAW_DIR
    / "barchart_wti_nearby_f1_f6_f12_2015_onward.parquet"
)
curve_path = (
    PROCESSED_DIR
    / "barchart_wti_curve_f1_f6_f12_2015_onward.parquet"
)
symbols_path = (
    PROCESSED_DIR
    / "barchart_wti_contract_symbols_2015_onward.parquet"
)

barchart_long.to_parquet(
    raw_path,
    index=False,
)
barchart_curve.to_parquet(
    curve_path,
    index=False,
)
barchart_symbols.to_parquet(
    symbols_path,
    index=False,
)

print("saved:", raw_path)
print("saved:", curve_path)
print("saved:", symbols_path)


## Final QA


In [ ]:
print("Barchart WTI source-layer build completed.")
print(
    "date range:",
    barchart_curve["trading_day"].min(),
    "->",
    barchart_curve["trading_day"].max(),
)
print("daily curve rows:", len(barchart_curve))
print("raw long rows:", len(barchart_long))
print(
    "nearby series:",
    sorted(
        barchart_long["nearby"]
        .unique()
        .tolist()
    ),
)
print(
    "complete curve share:",
    round(complete_share, 4),
)

for path in (
    raw_path,
    curve_path,
    symbols_path,
):
    assert path.exists(), path

print("All Barchart outputs: OK")


## Next step: validate against EIA

Compare Barchart F1–F4 with EIA RCLC1–RCLC4 over their shared
2015–2024 period before using Barchart as the final futures source.
